In [1]:
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [4]:
loader = PyPDFLoader("../utils/Society_Maintenance_Tracker.pdf")
docs = loader.load()
len(docs)

2

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)

len(splitted_data)

4

In [6]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [7]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [8]:
query = "What this society maintenance tracker ?"
data = vector_store.similarity_search(query=query)

In [11]:
context = ""
for doc in data:
    context += doc.page_content + "\n"

print(context)

Society Maintenance Tracker
Objective:
• Apartment societies handle a steady stream of maintenance complaints, but without a proper
system the admin has no way to see what is pending, what is overdue, or which issues keep
coming back. Residents, meanwhile, have no visibility into progress. Build a platform where
residents raise and track complaints with supporting photos, the admin manages them through a
clear workflow with priorities, and everyone stays informed through a notice board and email
updates.
Scope of Work:
• Input: Resident complaints with photos, admin status and priority updates, admin notices
• Output: Tracked complaints with history, a notice board, and email updates to residents
• Resident can register, log in, and raise a complaint with a category, description, and optional
photo
• Resident can view all their complaints with the full status history of each
• Admin can view all complaints, filter by category, status, or date, and set a priority (Low,
Medium, High)
• O

In [12]:
llm = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [13]:
res = llm.invoke(f"Can you provide me the answer based on provided context for my question, context: {context} and question: {query}")

In [14]:
print(res.content)

**Society Maintenance Tracker** is a web‑based platform designed to streamline how apartment societies handle maintenance requests.  
It gives residents a simple way to file complaints (with optional photos), view the full history of each request, and receive email updates when the status changes.  
Administrators can see every complaint, filter and prioritize them, update their status (Open → In‑Progress → Resolved), flag overdue items, and post notices to a public board.  
The system automatically detects overdue complaints based on a configurable threshold, pins important notices, and sends email notifications to keep everyone informed.  
In short, it turns a chaotic, manual complaint process into a transparent, auditable, and efficient workflow for both residents and society staff.


#### Chain - Context_generate | prompt | llm | strparser

In [21]:
def get_context(query: str):
    data = vector_store.similarity_search(query=query)

    context = ""

    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context": context,
        "question": query
    }

In [22]:
prompt = PromptTemplate.from_template(
    """
    You are a helpful assistant. Provide an answer based on the context
    for the user's question.

    If you don't know the answer, say "I don't know."

    Context:
    {context}

    Question:
    {question}
    """
)

In [23]:
rag_chain = get_context | prompt | llm

In [24]:
res = rag_chain.invoke(
    "What is the tech stack that can be used to make the tracker?"
)

print(res.content)

## Suggested Tech Stack for the Apartment Society Maintenance Tracker

| Layer | Recommended Technology | Why it fits the requirements |
|-------|------------------------|------------------------------|
| **Frontend (Web)** | **React** (Create‑React‑App / Vite) + **Tailwind CSS** | • Component‑driven UI for resident & admin dashboards.<br>• Tailwind gives quick, responsive styling and a clean design system.<br>• React Router for page navigation (login, complaints, notice board, admin panel). |
| **Frontend (Mobile)** | **React Native** (Expo) or **Flutter** | • If you want a native‑looking mobile app for residents.<br>• Shares most logic with the web app (API calls, auth). |
| **Backend (API)** | **Node.js + Express** (or **NestJS** for a more opinionated framework) | • Mature ecosystem, easy to add middleware for auth, file uploads, and email.<br>• Works well with a JSON‑based REST API that the React front‑end can consume. |
| **Auth & Role Management** | **JWT** (JSON Web Tokens) + *